In [1]:
import pandas as pd
from collections import Counter

In [2]:
# FUNGSI DAN PROSEDUR #

# Buat Model Naive Bayes
def naiveBayes(x, y):
    model = {}
    output = Counter(y)
    N = len(y)
    probabilitas_fitur = {}  #Likelihood

    model["prior"] = {
        k: output[k]/N
        for k in output
    }

    for k in output:
        probabilitas_fitur[k] = {}
        x_k = [x[i] for i in range(len(x)) if y[i] == k]  #ambil semua data dengan output k

        for f in range(len(x[0])):
            probabilitas_fitur[k][f] = {}  #kelas K, fitur f
            fitur_f = [xf[f] for xf in x_k]  #ambil semua fitur f dari kelas k
            kategori_count = Counter(fitur_f)  #hitung jumlah kemunculan setiap kategori
            n_kategori = len(fitur_f)

            for v in kategori_count:
                probabilitas_fitur[k][f][v] = kategori_count[v] / n_kategori  #kelas K, fitur f, probabilitas v

    model["likelihood"] = probabilitas_fitur
    model["jumlah_kelas"] = output
    model["jumlah_data"] = N

    return model

In [3]:
# Prediksi
def prediksi_naiveBayes(model, x):
    probabilitas = {}

    # Menghitung prior x likelihood untuk setiap kelas
    for k in model["prior"]:
        probabilitas_x = model["prior"][k]

        for f in range(len(x)):
            likelihood = model["likelihood"][k][f].get(x[f], 1e-6)
            probabilitas_x *= likelihood

        probabilitas[k] = probabilitas_x

    # Normalisasi agar menjadi probabilitas posterior
    total = sum(probabilitas.values())

    if total == 0:
        posterior = {k: 0 for k in probabilitas}
    else:
        posterior = {
            k: probabilitas[k] / total
            for k in probabilitas
        }

    # Kelas dengan probabilitas terbesar
    kelas_prediksi = max(posterior, key=posterior.get)

    return kelas_prediksi, probabilitas, posterior

In [4]:
# inisialisasi data
df = pd.read_excel("../dataset/weather_dataset.xlsx")
fitur = ["Outlook", "Temperatur", "Humidity", "Wind"]

target = "Play"

# memastikan data tidak memiliki nilai kosong
df = df[fitur + [target]].dropna().copy()

# membersihkan spasi pada data kategorikal
for kolom in fitur + [target]:
    df[kolom] = df[kolom].astype(str).str.strip()

    # memisahkan fitur dan target
    x = df[fitur].values.tolist()
    y = df[target].values.tolist()

In [5]:
# MAIN PROGRAM #

model_naiveBayes = naiveBayes(x, y)

x_uji = ["Rain", "Hot", "High", "Weak"]
print("\nData uji:")
for i in range(len(fitur)):
    print(fitur[i], "=", x_uji[i])

prediksi, probabilitas, posterior = prediksi_naiveBayes(
    model_naiveBayes,
    x_uji
)

print("\nProbabilitas:")
for kelas, p in posterior.items():
    print(f"Play = {kelas}: {p:.4%}")

print("\nkelas hasil prediksi:", prediksi)


Data uji:
Outlook = Rain
Temperatur = Hot
Humidity = High
Wind = Weak

Probabilitas:
Play = No: 51.6334%
Play = Yes: 48.3666%

kelas hasil prediksi: No
